In [5]:
import pandas as pd
# Lendo o arquivo Parquet
df = pd.read_parquet('../data/processed/bairros_features.parquet')

print(df)




               bairro codbairro          regiao_adm  codra   area_km2  \
0    FREGUESIA (ILHA)       098  ILHA DO GOVERNADOR     20   4.056414   
1           BANCARIOS       097  ILHA DO GOVERNADOR     20   0.978049   
2              GALEAO       104  ILHA DO GOVERNADOR     20  18.957473   
3                TAUA       101  ILHA DO GOVERNADOR     20   1.672550   
4          PORTUGUESA       103  ILHA DO GOVERNADOR     20   1.186413   
..                ...       ...                 ...    ...        ...   
160             IRAJA       076               IRAJA     14   7.481982   
161      BRAS DE PINA       045               PENHA     11   2.956169   
162      VISTA ALEGRE       075               IRAJA     14   1.254362   
163     VILA DA PENHA       074               IRAJA     14   1.541675   
164         ARGENTINO       166               PENHA     11   0.031769   

     populacao  domicilios  densidade_hab_km2    ips  n_paradas  n_linhas  \
0      15900.0      7629.0        3919.718488 

Aqui obtive uma visão geral dos dados existentes para cada bairro.

In [6]:
dfdemanda = df[['densidade_hab_km2', 'ips']]
dfdemanda['ips'] = 1-dfdemanda['ips']/100
dfdemanda = dfdemanda.rename(columns={'ips': 'ips_invertido'})
dfdemanda['densidade_hab_km2'] = (df['densidade_hab_km2'] - df['densidade_hab_km2'].min()) / (df['densidade_hab_km2'].max() - df['densidade_hab_km2'].min())
dfdemanda['ips_invertido'] = (dfdemanda['ips_invertido'] - dfdemanda['ips_invertido'].min()) / (dfdemanda['ips_invertido'].max() - dfdemanda['ips_invertido'].min())
print(dfdemanda)

     densidade_hab_km2  ips_invertido
0             0.079105       0.488464
1             0.228988       0.488464
2             0.021880       0.488464
3             0.320717       0.488464
4             0.338600       0.488464
..                 ...            ...
160           0.232465       0.544056
161           0.308657       0.626675
162           0.113740       0.544056
163           0.288358       0.544056
164                NaN       0.626675

[165 rows x 2 columns]


Aqui criei uma planilha contendo os indicadores de demanda normalizados.

In [7]:
dfidemanda = (dfdemanda['densidade_hab_km2'] + dfdemanda['ips_invertido'])/2
print(dfidemanda)

0      0.283784
1      0.358726
2      0.255172
3      0.404590
4      0.413532
         ...   
160    0.388261
161    0.467666
162    0.328898
163    0.416207
164         NaN
Length: 165, dtype: float64


Aqui fiz a média dos indicadores de demanda, formando um índice único. Vou fazer o mesmo para formar o índice de oferta.

In [8]:
dfoferta = df[['cobertura_400m', 'n_paradas', 'n_linhas', 'headway_medio_min']]
dfoferta = dfoferta.rename(columns={'n_paradas': 'paradas_km2'})
dfoferta = dfoferta.rename(columns={'headway_medio_min': 'headway_inv'})
dfoferta['paradas_km2'] = df['n_paradas']/df['area_km2']
dfoferta['headway_inv'] = 1/df['headway_medio_min']
dfoferta['paradas_km2'] = (dfoferta['paradas_km2'] - dfoferta['paradas_km2'].min()) / (dfoferta['paradas_km2'].max() - dfoferta['paradas_km2'].min())
dfoferta['headway_inv'] = (dfoferta['headway_inv'] - dfoferta['headway_inv'].min()) / (dfoferta['headway_inv'].max() - dfoferta['headway_inv'].min())
dfoferta['n_linhas'] = (df['n_linhas'] - df['n_linhas'].min()) / (df['n_linhas'].max() - df['n_linhas'].min())
dfoferta['cobertura_400m'] = (df['cobertura_400m'] - df['cobertura_400m'].min()) / (df['cobertura_400m'].max() - df['cobertura_400m'].min())
dfioferta = dfoferta.sum(axis=1)/4

print(dfioferta)

0      0.225637
1      0.437132
2      0.212067
3      0.423253
4      0.441218
         ...   
160    0.510962
161    0.479744
162    0.533192
163    0.457509
164    0.250000
Length: 165, dtype: float64


Montagem do índice:

In [9]:
DTI = (dfidemanda - dfioferta + 1)*50
print(DTI)

0      52.907360
1      46.079707
2      52.155271
3      49.066865
4      48.615671
         ...    
160    43.864936
161    49.396114
162    39.785318
163    47.934941
164          NaN
Length: 165, dtype: float64


In [10]:
DTI.describe()

count    164.000000
mean      48.869670
std        8.387979
min       26.194849
25%       45.200964
50%       48.585529
75%       52.936400
max       79.754472
dtype: float64

In [11]:
df['DTI'] = DTI
print("DESERTOS (DTI alto):")
print(df.nlargest(15, 'DTI')[['bairro', 'DTI', 'ips', 'cobertura_400m', 'densidade_hab_km2']])  

DESERTOS (DTI alto):
                 bairro        DTI    ips  cobertura_400m  densidade_hab_km2
141             ROCINHA  79.754472  51.16        0.757832       49327.908563
153            GERICINO  70.704596  54.34        0.115072        8163.954383
73          JACAREZINHO  70.595253  43.56        1.000000       37449.809756
17                ACARI  68.105763  41.58        0.694701       18481.763515
42   COMPLEXO DO ALEMAO  66.886178  47.14        0.499113       18306.148484
19         COSTA BARROS  66.171470  41.58        0.883598       14476.234601
10               PAVUNA  63.255973  41.58        0.871970       11470.592016
123      CIDADE DE DEUS  63.141546  46.12        0.871629       24018.321387
12      PARQUE COLUMBIA  63.061392  41.58        0.837081        5419.412778
31                 MARE  61.512845  50.24        0.806061       29243.065061
53           SANTA CRUZ  61.460225  49.57        0.264740        1992.328674
40            PACIENCIA  60.874303  49.57        0.4407

In [12]:
# Bottom 15 (= bem servidos esperados)
print("\nBEM SERVIDOS (DTI baixo):")
print(df.nsmallest(15, 'DTI')[['bairro', 'DTI', 'ips', 'cobertura_400m', 'densidade_hab_km2']])


BEM SERVIDOS (DTI baixo):
                bairro        DTI    ips  cobertura_400m  densidade_hab_km2
101  PRACA DA BANDEIRA  26.194849  76.81        1.000000       10022.063403
104             GLORIA  28.619458  87.09        0.939872        6245.238544
134              LAGOA  29.206551  85.86        0.883547        3654.861156
130    JARDIM BOTANICO  31.179278  85.86        0.837368        5901.731345
119        COSME VELHO  31.335111  87.09        0.880580        7179.490295
121           BOTAFOGO  31.772069  87.09        0.918198       16048.865055
88              CENTRO  32.027155  58.15        0.840420        4358.170205
138             LEBLON  32.031852  85.86        0.993454       17513.803843
136              GAVEA  33.364104  85.86        0.710086        5495.333160
100        CIDADE NOVA  33.901729  55.40        1.000000        5075.651405
140        SAO CONRADO  34.794870  85.86        0.616652        1500.944047
103           MARACANA  35.138529  73.73        1.000000     

In [13]:
import geopandas

df['DTI'] = DTI

# Excluir bairro ARGENTINO (sem populacao no Censo)
df = df.dropna(subset=['DTI'])

# Convert the DataFrame to a GeoDataFrame before saving as GeoJSON
df = geopandas.GeoDataFrame(df, geometry=geopandas.GeoSeries.from_wkb(df['geometry']))

# Salvar pra Etapa 4 consumir
df.to_parquet('../data/processed/bairros_dti.parquet')
df.to_file('../data/processed/bairros_dti.geojson', driver='GeoJSON')

print(f"✓ Salvo: {len(df)} bairros com DTI calculado")

/mnt/c/Users/camil/codes/analytica/PS-2026.1-DesertoTransporte/.venv/lib/python3.12/site-packages/pyogrio/geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


✓ Salvo: 164 bairros com DTI calculado


Estarei montando um gráfico com todos os DTI's a mostra para ter uma melhor noção